# Episode 5: Deploy a GPT: Hugging Face, Gradio, a browser

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep005-deploy.ipynb)

We save our model as a GPT-2 and push it to the Hugging Face Hub, with a tag per episode, serve it from a Gradio app on a VPS, and run it in the browser with Transformers.js.

[The video](https://youtu.be/rn72Y_9dVE0) · [The episode's page](https://mvgpt.nenuadrian.com/ep005-deploy/) · [The docs](https://nenuadrian.github.io/mvGPT/deploy/) · [The code at `ep005/end`](https://github.com/nenuadrian/mvGPT/tree/ep005/end)

This notebook runs the episode's commands on its code, as the episode left it. Models and runs from the episodes come from [the Hub](https://huggingface.co/nenuadrian/mvGPT-checkpoints), the files the videos used. It runs on a GPU runtime, and Colab's free T4 will do.

In [ ]:
# The code as the episode left it, at its last tag, and what it needs.
import os

if not os.path.isdir("/content/mvGPT"):
    !git -c advice.detachedHead=false clone --quiet --depth 1 --branch ep005/end https://github.com/nenuadrian/mvGPT /content/mvGPT
%cd /content/mvGPT
!pip install --quiet -r requirements.txt

import torch

print(torch.cuda.get_device_name() if torch.cuda.is_available() else "No GPU: Runtime > Change runtime type > T4 GPU")

from huggingface_hub import hf_hub_download


def fetch(*names):
    # Models from the episodes, from the Hub: the files the videos used, where the scripts look for them.
    for name in names:
        print(hf_hub_download("nenuadrian/mvGPT-checkpoints", f"checkpoints/{name}/model.pt", local_dir="."))


import shlex
import subprocess
import time
import urllib.error
import urllib.parse
import urllib.request


def start(command, url):
    # A server in the background, once it answers at url; its output goes to /tmp/server-<port>.log.
    log = open(f"/tmp/server-{urllib.parse.urlparse(url).port}.log", "w")
    server = subprocess.Popen(shlex.split(command), stdout=log, stderr=subprocess.STDOUT)
    while True:
        try:
            urllib.request.urlopen(url, timeout=1)
            return server
        except urllib.error.HTTPError:
            return server
        except OSError:
            if server.poll() is not None:
                raise RuntimeError(open(log.name).read())
            time.sleep(1)


def stop(server):
    server.terminate()
    server.wait()


fetch("pretrain", "sft")

## GPT-2's config
Our model has GPT-2's architecture, so transformers can load it as one.

In [ ]:
import json
import urllib.request

config = json.load(urllib.request.urlopen("https://huggingface.co/openai-community/gpt2/resolve/main/config.json"))
print(json.dumps({k: config[k] for k in ("model_type", "n_layer", "n_head", "n_embd", "n_positions", "vocab_size", "activation_function")}, indent=2))

## Saving it as a GPT-2, and as ONNX
`hub.py` saves the weights under GPT-2's names with a tokenizer.json, checks that transformers
gives our logits, and exports it to ONNX, also quantized to 8 bits.

In [ ]:
!python hub.py checkpoints/sft/model.pt hf/sft 2>/dev/null
!ls hf/sft hf/sft/onnx

## Pushing it to the Hub
The episode pushes to nenuadrian/mvGPT, with a tag per episode. To push to your own account,
log in with a token that can write, and put your name in the repo's.

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()
# !python hub.py checkpoints/sft/model.pt hf/sft --push YOUR_NAME/mvGPT --tag ep003-sft --message "Fine-tuned on TinyStories-Instruct"

## The app
The Gradio app loads each episode's version from the Hub and writes stories. It runs here in
the background, and Colab shows it through its proxy.

In [ ]:
from google.colab import output

app = start("python app.py", "http://127.0.0.1:7860")
output.serve_kernel_port_as_iframe(7860, height=720)

## In the browser
`web/index.html` runs the model in the browser with Transformers.js: the page downloads the
8-bit ONNX model from the Hub when you ask for a story.

In [ ]:
page = start("python -m http.server 8001 --directory web", "http://127.0.0.1:8001")
output.serve_kernel_port_as_iframe(8001, height=720)

## On a server
The episode deploys the app in Docker to a VPS over SSH, with `./deploy.sh user@host`: that
takes a server of your own, so it isn't run here.

Next: [episode 6, Profiling a GPT with PyTorch's profiler](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep006-profiling.ipynb)